<a href="https://colab.research.google.com/github/anyuanay/info212/blob/main/INFO212_Week9_Lecture_examples_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 212: Data Science Programming 1
___

### Week 9: Data Analysis Examples

---

**Question:**
- What can I learn from real world data analysis examples? 

**Objectives:**
- Apply the techiques learned in this course to real world data analysis problems

In [ ]:
import numpy as np
import pandas as pd
PREVIOUS_MAX_ROWS = pd.options.display.max_rows
pd.options.display.max_rows = 20
np.random.seed(12345)
import matplotlib.pyplot as plt
plt.rc('figure', figsize=(10, 6))
np.set_printoptions(precision=4, suppress=True)

# Data Wrangling on Names for the Titanic Data
1. Extract titles from names.
2. Consolidate titles to a small list of common titles.
3. Add a new column indiating the titles of passergers.
4. Convert the titles columns to numeric values.

In [ ]:
from google.colab import files
files.upload()

In [ ]:
df = pd.read_csv("train.csv")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.Name

In [ ]:
import re
pattern = r' ([A-Za-z]+)\.'

In [ ]:
regx = re.compile(pattern, flags=re.IGNORECASE)

In [ ]:
df.iloc[3].Name

## Review the differences between the regular expression function:
- findall()
- match()
- search()

In [ ]:
regx.findall(df.iloc[3].Name)

In [ ]:
# for na in list(df.Names):

In [ ]:
df.Name.isnull().sum()

In [ ]:
df.Name.str.findall(pattern)

In [ ]:
df.Name.str.extract(pattern, expand=False)

In [ ]:
df.Name.str.extract(pattern, expand=False)

In [ ]:
df['Title'] = df.Name.str.extract(pattern, expand=False)

In [ ]:
df.head()

In [ ]:
df.Title.value_counts()

In [ ]:
pd.crosstab(df['Title'], df['Sex']).plot.bar()

We can replace many titles with a more common name or classify them as `Rare`.

In [ ]:
title_repl = {'Col':'Mr', 'Major':'Master', 'Mlle':'Miss', 'Mme':'Mrs', 'Sir':'Mr', 'Capt':'Master', 'Countess':'Mrs', 
'Jonkheer':'Mr', 'Don':'Master', 'Ms':'Miss', 'Lady':'Miss'}

In [ ]:
df['Title_repl'] = df['Title'].replace(title_repl)

In [ ]:
df.head()

In [ ]:
df[df.Title == 'Col']

In [ ]:
df['Title_repl'].unique()

In [ ]:
df['Title_repl'].value_counts().plot.bar()

In [ ]:
pd.crosstab(df.Title_repl, df.Survived).plot.bar()

In [ ]:
df[['Title_repl', 'Survived']].groupby(['Title_repl'], as_index=False).mean()

We can convert the categorical titles to ordinal.

In [ ]:
title_mapping = {'Dr':0, 'Master':1, 'Miss':2, 'Mr':3, 'Mrs':4, 'Rev':5}

In [ ]:
df['Title_num'] = df['Title_repl'].map(title_mapping)

In [ ]:
df['Title_num']

In [ ]:
df.head()

In [ ]:
df['Title_num'].isnull().sum()

In [ ]:
df.head(3)

Now we can safely drop the Name feature from the dataset. We also do not need the PassengerId feature in the dataset.

In [ ]:
df = df.drop(['Name', 'PassengerId'], axis=1)

In [ ]:
df.head(3)

## USDA Food Database
The US Department of Agriculture makes available a database of food nutrient information. 
The records look like this:

```
{
  "id": 21441,
  "description": "KENTUCKY FRIED CHICKEN, Fried Chicken, EXTRA CRISPY,
Wing, meat and skin with breading",
  "tags": ["KFC"],
  "manufacturer": "Kentucky Fried Chicken",
  "group": "Fast Foods",
  "portions": [
    {
      "amount": 1,
      "unit": "wing, with skin",
      "grams": 68.0
    },

    ...
  ],
  "nutrients": [
    {
      "value": 20.8,
      "units": "g",
      "description": "Protein",
      "group": "Composition"
    },

    ...
  ]
}
```

Each food has a number of identifying attributes along with two lists of nutrients and portion sizes. Data in this form is not particularly amenable to analysis, so we need to do some work to wrangle the data into a better form.

```
import json
db = json.load(open('datasets/usda-food-database.json'))
len(db)
```

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import json
db = json.load(open('usda-food-database.json'))

In [ ]:
len(db)

In [ ]:
db[1]

Each entry in db is a dict containing all the data for a single food. The 'nutrients'
field is a list of dicts, one for each nutrient:

```
import pandas as pd
db[0].keys()
```

In [ ]:
db[1].keys()

```
db[0]['nutrients'][0]
```

In [ ]:
db[0]['description']

In [ ]:
len(db[0]['nutrients'])

```
nutrients = pd.DataFrame(db[0]['nutrients'])
nutrients.head()
```

In [ ]:
len(db[0]['nutrients'])

In [ ]:
import pandas as pd
nutrients = pd.DataFrame(db[0]['nutrients'])

In [ ]:
nutrients.head()

In [ ]:
nutrients.shape

When converting a list of dicts to a DataFrame, we can specify a list of fields to
extract. We’ll take the food names, group, ID, and manufacturer:

```
info_keys = ['description', 'group', 'id', 'manufacturer']
```

In [ ]:
info_keys = ['description', 'group', 'id', 'manufacturer']

```
info = pd.DataFrame(db, columns=info_keys)
info.head()
```

In [ ]:
db[0].keys()

In [ ]:
info = pd.DataFrame(db, columns = info_keys)

In [ ]:
info.head()

In [ ]:
info.shape

```
info.info()```

In [ ]:
info.info()

You can see the distribution of food groups with value_counts:

```
pd.value_counts(info.group)[:10]
```

In [ ]:
info.group.unique()

In [ ]:
pd.value_counts(info.group)[:10]

In [ ]:
info['group'].nunique()

In [ ]:
pd.value_counts(info.group)[:10]

Now, to do some analysis on all of the nutrient data, it’s easiest to assemble the
nutrients for each food into a single large table. To do so, we need to take several
steps. First, I’ll convert each list of food nutrients to a DataFrame, add a column for
the food id, and append the DataFrame to a list. Then, these can be concatenated
together with concat:

```
nutrients = []

for rec in db:
    fnuts = pd.DataFrame(rec['nutrients'])
    fnuts['id'] = rec['id']
    nutrients.append(fnuts)

nutrients = pd.concat(nutrients, ignore_index=True)
```

In [ ]:
nutrients = []
for rec in db:
    fnuts = pd.DataFrame(rec['nutrients'])
    fnuts['id'] = rec['id']
    nutrients.append(fnuts)

In [ ]:
len(nutrients)

In [ ]:
nutrients[2].head()

In [ ]:
nutrients[0].head()

In [ ]:
nutrients_df = pd.concat(nutrients, ignore_index = True)

In [ ]:
nutrients_df.sample(10)

In [ ]:
nutrients_df.shape

In [ ]:
389355/6636

Check duplicates:

```
nutrients.duplicated().sum()  # number of duplicates
```

In [ ]:
nutrients_df.duplicated().sum()

```
nutrients = nutrients.drop_duplicates()
```

In [ ]:
nutrients_nodup = nutrients_df.drop_duplicates()

In [ ]:
info.columns

In [ ]:
nutrients_nodup.columns

Since 'group' and 'description' are in both DataFrame objects, we can rename for
clarity:

```
col_mapping = {'description' : 'food',
               'group'       : 'fgroup'}
```

In [ ]:
col_mapping = {'description': 'food', 'group': 'fgroup'}

```
info = info.rename(columns=col_mapping, copy=False)
info.info()
```

In [ ]:
info = info.rename(columns = col_mapping, copy = False)

In [ ]:
info.head()

In [ ]:
info.columns

```
col_mapping = {'description' : 'nutrient',
               'group' : 'nutgroup'}
nutrients = nutrients.rename(columns=col_mapping, copy=False)
nutrients.head()
```

In [ ]:
col_mapping = {'description': 'nutrient', 'group': 'nutgroup'}

In [ ]:
nutrients_clean = nutrients_nodup.rename(columns = col_mapping,copy = False)

In [ ]:
nutrients_clean.head()

With all of this done, we’re ready to merge info with nutrients:

```
ndata = pd.merge(nutrients, info, on='id', how='outer')
ndata.info()
```

In [ ]:
ndata = pd.merge(nutrients_clean, info, on = 'id', how = 'outer')

In [ ]:
ndata.head()

In [ ]:
nutrients_clean.shape

In [ ]:
ndata.shape

```
ndata.iloc[30000]
```

In [ ]:
ndata.iloc[30000]

We could now make a plot of median values by food group and nutrient type

In [ ]:
maxv = ndata[ndata.nutrient == 'Protein'].value.max()

In [ ]:
ndata[(ndata.nutrient == 'Protein') & (ndata.value == maxv)]

```
fig = plt.figure()
```

In [ ]:
fig = plt.figure()

```
result = ndata.groupby(['nutrient', 'fgroup'])['value'].quantile(0.5)
```

In [ ]:
result = ndata.groupby(['nutrient', 'fgroup'])['value'].quantile(0.5)

In [ ]:
ndata.groupby(['nutrient', 'fgroup'])['value'].quantile(0.5)

In [ ]:
result.head()

```
result['Zinc, Zn'].sort_values().plot(kind='barh')
```

In [ ]:
%matplotlib inline
result['Protein'].sort_values().plot(kind = 'barh')

We can find which food is most dense in each nutrient:

```
by_nutrient = ndata.groupby(['nutgroup', 'nutrient'])

get_maximum = lambda x: x.loc[x.value.idxmax()]
get_minimum = lambda x: x.loc[x.value.idxmin()]

max_foods = by_nutrient.apply(get_maximum)[['value', 'food']]

# make the food a little smaller
max_foods.food = max_foods.food.str[:50]
```

In [ ]:
by_nutrient  = ndata.groupby(['nutgroup', 'nutrient'])

In [ ]:
by_nutrient.count()

In [ ]:
get_min = lambda x: x.loc[x.value.idxmin()]

In [ ]:
get_min

In [ ]:
get_min = lambda x: x.loc[x.value.idxmin()]

In [ ]:
get_min

In [ ]:
get_maximum

In [ ]:
max_foods = by_nutrient.apply(get_maximum)[['value', 'food']]

In [ ]:
min_foods = by_nutrient.apply(get_min)

In [ ]:
min_foods.loc['Amino Acids']

In [ ]:
max_foods

In [ ]:
max_foods.food = max_foods.food.str[:50]

In [ ]:
max_foods.food[:10]

```
max_foods.loc['Amino Acids']['food']
```

In [ ]:
max_foods.loc['Amino Acids']['food']